Transformation → Silver
Source → Bronze (bronze.crm_interactions)
Target → silver.crm_interactions

1. Read Bronze

In [0]:
df_bronze = spark.read.table("bronze.crm_interactions")

2.Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

2. Flatten JSON

In [0]:
df_flat = df_bronze.select(
    F.col("interaction_id"),
    F.col("customer_id"),
    F.col("interaction_type"),
    F.col("channel"),

    # rename timestamp → align naming
    F.col("timestamp").alias("interaction_timestamp"),

    # nested
    F.col("details.amount").alias("amount"),
    F.col("location.city").alias("city"),
    F.col("location.country").alias("country"),
    F.col("metadata.source").alias("source"),

    # system
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
display(df_flat)

3. Data Type Casting

In [0]:
df_typed = df_flat \
    .withColumn("interaction_id", F.col("interaction_id").cast("string")) \
    .withColumn("customer_id", F.col("customer_id").cast("string")) \
    .withColumn("interaction_type", F.col("interaction_type").cast("string")) \
    .withColumn("channel", F.col("channel").cast("string")) \
    .withColumn("interaction_timestamp", F.to_timestamp("interaction_timestamp")) \
    .withColumn("amount", F.col("amount").cast("double")) \
    .withColumn("city", F.col("city").cast("string")) \
    .withColumn("country", F.col("country").cast("string")) \
    .withColumn("source", F.col("source").cast("string")) \
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))

4. Handle Nulls

In [0]:
df_clean = df_typed.dropna(subset=["interaction_id", "customer_id"])

5. Deduplication

In [0]:

window_spec = Window.partitionBy("interaction_id").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = df_clean \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")

 Key Standardization

In [0]:

# Standardize customer_id format across systems


from pyspark.sql.functions import regexp_replace, col

df_standardized = (
    df_dedup
    .withColumn("customer_id", regexp_replace(col("customer_id"), "CUST_", ""))
    .withColumn("customer_id", regexp_replace(col("customer_id"), "^0+", ""))
    .filter(col("customer_id") != "")
)

Validation

In [0]:
df_valid = df_standardized.filter(
    F.col("customer_id").isNotNull()
)

6. Data Quality Rules

In [0]:
df_valid = df_standardized.filter(
    F.col("customer_id").isNotNull()
)


7. Derived Columns

In [0]:
df_enriched = df_valid \
    .withColumn("interaction_year", F.year("interaction_timestamp")) \
    .withColumn("interaction_month", F.month("interaction_timestamp"))

In [0]:
df_enriched.select("customer_id").show(10, False)

8. Write to Silver (MERGE)

In [0]:
from delta.tables import DeltaTable

table_name = "silver.crm_interactions"

if spark.catalog.tableExists(table_name):

    delta_table = DeltaTable.forName(spark, table_name)

    delta_table.alias("target").merge(
        df_enriched.alias("source"),
        "target.interaction_id = source.interaction_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    df_enriched.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

9. Validation

In [0]:
spark.read.table("silver.crm_interactions").count()
display(spark.read.table("silver.crm_interactions"))

In [0]:
spark.read.table("silver.crm_interactions").count()